# 04j — Weighted SupCon + Learned Class Proxies

## Obiettivo

Migliorare la discriminazione delle 12 classi Tumor + Healthy combinando:

1. Supervised Contrastive Learning;
2. classificazione cosine-based tramite proxy di classe apprendibili;
3. pesi di classe per compensare lo sbilanciamento;
4. utilizzo dell'intero training set senza downsampling statico.

Il modello viene inizializzato dal best checkpoint SupCon V3 ottenuto
nell'esperimento 04i.

## Motivazione

SupCon ha migliorato la classificazione rispetto alla Euclidean V3,
ma l'accuracy rimane circa al 30%.

Nel training precedente ogni epoca utilizzava circa 2104 sample per
classe, limitando fortemente l'utilizzo delle classi più numerose.

In questo esperimento tutti i sample train vengono utilizzati nella
loro distribuzione reale.

Lo sbilanciamento viene compensato mediante class weights basati
sull'effective number of samples.

## Architettura

FCGR
  ↓
CNN V3
  ↓
embedding h (128D)
  ├── projection head → z → SupCon Loss
  │
  └── cosine similarity → 12 learned class proxies
                              ↓
                        Weighted CE

Loss totale:

    L = WeightedCE + λ * SupCon

con λ = 0.25.

I proxy rappresentano direzioni apprendibili nello spazio embedding
associate alle 12 classi.

Durante l'inferenza un nuovo sample può quindi essere associato alle
classi mediante cosine similarity tra il suo embedding e i proxy.

## Task

- 11 classi tumorali
- Healthy
- FCGR k=6
- 12 classi totali
- test set non utilizzato

In [1]:
# ============================================================
# CELL 2 — IMPORT + CONFIG
# ============================================================

from pathlib import Path

import json
import random
import time
import copy
import gc

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import (
    Dataset,
    DataLoader
)

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    balanced_accuracy_score,
    confusion_matrix,
    classification_report
)


# ============================================================
# PATH
# ============================================================

CURRENT_DIR = Path.cwd().resolve()

if CURRENT_DIR.name == "notebooks":
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    PROJECT_ROOT = CURRENT_DIR


PROCESSED_DIR = (
    PROJECT_ROOT
    / "data"
    / "processed"
)


ARTIFACTS_DIR = (
    PROJECT_ROOT
    / "artifacts"
    / "weighted_supcon_proxy_tumor_healthy"
)

ARTIFACTS_DIR.mkdir(
    parents=True,
    exist_ok=True
)


MANIFEST_PATH = (
    PROCESSED_DIR
    / "siamese_tumor_healthy_manifest.tsv"
)


CLASS_MAPPING_PATH = (
    PROCESSED_DIR
    / "siamese_tumor_healthy_class_mapping.tsv"
)


VAL_POOL_PATH = (
    PROCESSED_DIR
    / "siamese_val_pair_pool.tsv"
)


PAIR_CONFIG_PATH = (
    PROCESSED_DIR
    / "siamese_pair_config.json"
)


SUPCON_CHECKPOINT_PATH = (
    PROJECT_ROOT
    / "artifacts"
    / "supcon_v3_tumor_healthy"
    / "supcon_v3_tumor_healthy_best.pt"
)


with open(
    PAIR_CONFIG_PATH,
    "r",
    encoding="utf-8"
) as f:

    pair_config = json.load(f)


K = int(
    pair_config["k"]
)

RANDOM_STATE = int(
    pair_config["random_state"]
)


FCGR_PATH = (
    PROCESSED_DIR
    / "fcgr_cache"
    / f"fcgr_k{K}.npy"
)


FCGR_INDEX_PATH = (
    PROCESSED_DIR
    / "fcgr_cache"
    / f"fcgr_k{K}_index.tsv"
)


# ============================================================
# MODEL CONFIG
# ============================================================

N_CLASSES = 12

EMBEDDING_DIM = 128

PROJECTION_DIM = 128

TEMPERATURE = 0.07


# peso SupCon nella loss combinata
SUPCON_WEIGHT = 0.25


# Cosine classifier scale
PROXY_SCALE = 16.0


# Effective-number weighting
CLASS_WEIGHT_BETA = 0.9999


# ============================================================
# TRAINING
# ============================================================

TRAIN_BATCH_SIZE = 128

EVAL_BATCH_SIZE = 256


BACKBONE_LR = 1e-4

PROXY_LR = 5e-4

WEIGHT_DECAY = 1e-4


# ============================================================
# DEVICE
# ============================================================

def set_seed(seed):

    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


set_seed(
    RANDOM_STATE
)


DEVICE = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)


AMP_ENABLED = (
    DEVICE.type == "cuda"
)


if DEVICE.type == "cuda":

    torch.backends.cudnn.benchmark = True

    torch.set_float32_matmul_precision(
        "high"
    )


print(
    "Device:",
    DEVICE
)

print(
    "k:",
    K
)

print(
    "SupCon checkpoint:",
    SUPCON_CHECKPOINT_PATH.exists()
)

print(
    "SupCon weight:",
    SUPCON_WEIGHT
)

print(
    "Proxy scale:",
    PROXY_SCALE
)

Device: cuda
k: 6
SupCon checkpoint: True
SupCon weight: 0.25
Proxy scale: 16.0


In [2]:
# ============================================================
# CELL 3 — FULL TRAIN + EFFECTIVE NUMBER CLASS WEIGHTS
# ============================================================

metadata = pd.read_csv(
    MANIFEST_PATH,
    sep="\t",
    dtype={"id": str}
)


metadata["class_id"] = (
    metadata["class_id"]
    .astype(int)
)


train_metadata = (
    metadata[
        metadata["split_cluster"]
        ==
        "train"
    ]
    .copy()
    .reset_index(drop=True)
)


# ============================================================
# VALIDATION
# ============================================================

class_mapping = pd.read_csv(
    CLASS_MAPPING_PATH,
    sep="\t"
)


old_to_new = dict(
    zip(
        class_mapping[
            "original_class_id"
        ].astype(int),

        class_mapping[
            "class_id"
        ].astype(int)
    )
)


val_original = pd.read_csv(
    VAL_POOL_PATH,
    sep="\t",
    dtype={"id": str}
)


val_original["class_id"] = (
    val_original["class_id"]
    .astype(int)
)


val_metadata = (
    val_original[
        val_original[
            "class_id"
        ].isin(
            old_to_new.keys()
        )
    ]
    .copy()
    .reset_index(drop=True)
)


val_metadata[
    "original_class_id"
] = (
    val_metadata[
        "class_id"
    ]
)


val_metadata[
    "class_id"
] = (
    val_metadata[
        "original_class_id"
    ]
    .map(old_to_new)
    .astype(int)
)


# ============================================================
# CLASS COUNTS
# ============================================================

class_counts = (

    train_metadata[
        "class_id"
    ]
    .value_counts()
    .sort_index()
    .reindex(
        range(N_CLASSES)
    )
    .to_numpy(
        dtype=np.float64
    )
)


# ============================================================
# EFFECTIVE NUMBER WEIGHTS
#
# w_c = (1-beta) / (1-beta^n_c)
# ============================================================

beta = CLASS_WEIGHT_BETA


effective_number = (
    1.0
    -
    np.power(
        beta,
        class_counts
    )
)


class_weights = (
    (1.0 - beta)
    /
    effective_number
)


# Normalizziamo media = 1
class_weights = (
    class_weights
    /
    class_weights.mean()
)


class_weights_tensor = torch.tensor(

    class_weights,

    dtype=torch.float32,

    device=DEVICE
)


weight_df = pd.DataFrame(
    {
        "class_id":
            np.arange(N_CLASSES),

        "n_train":
            class_counts.astype(int),

        "weight":
            class_weights
    }
)


weight_df = weight_df.merge(

    class_mapping[
        [
            "class_id",
            "disease_clean"
        ]
    ],

    on="class_id",

    how="left"
)


weight_df = weight_df[
    [
        "class_id",
        "disease_clean",
        "n_train",
        "weight"
    ]
]


print(
    "Train totale:",
    len(train_metadata)
)

print(
    "Validation:",
    len(val_metadata)
)

print()

display(
    weight_df
)

Train totale: 96167
Validation: 9753



,class_id,disease_clean,n_train,weight
0,0,gastric cancer,10000,0.730693
1,1,healthy,10000,0.730693
2,2,ovarian cancer,10000,0.730693
3,3,prostate cancer,10000,0.730693
4,4,colorectal cancer,10000,0.730693
5,5,lymphoma,10000,0.730693
6,6,cervical adenocarcinoma,10000,0.730693
7,7,leukemia,10000,0.730693
8,8,hypopharyngeal squamous cell carcinoma,5052,1.164559
9,9,glioblastoma cancer,4880,1.196128


In [3]:
# ============================================================
# CELL 4 — FCGR + NATURAL DISTRIBUTION LOADERS
# ============================================================

fcgr_memmap = np.load(
    FCGR_PATH,
    mmap_mode="r"
)


fcgr_index = pd.read_csv(
    FCGR_INDEX_PATH,
    sep="\t",
    dtype={"id": str}
)


id_to_fcgr_row = dict(
    zip(
        fcgr_index["id"],
        fcgr_index["fcgr_row"]
    )
)


class SingleFCGRDataset(Dataset):

    def __init__(
        self,
        metadata,
        fcgr_memmap,
        id_to_row
    ):

        self.metadata = (
            metadata
            .copy()
            .reset_index(drop=True)
        )


        self.fcgr_memmap = (
            fcgr_memmap
        )


        self.rows = (
            self.metadata["id"]
            .astype(str)
            .map(id_to_row)
            .to_numpy(dtype=np.int64)
        )


        self.labels = (
            self.metadata["class_id"]
            .to_numpy(dtype=np.int64)
        )


    def __len__(self):

        return len(
            self.metadata
        )


    def __getitem__(
        self,
        index
    ):

        row = int(
            self.rows[index]
        )


        fcgr = np.array(

            self.fcgr_memmap[row],

            dtype=np.float32,

            copy=True
        )


        return {

            "x":
                torch.from_numpy(
                    fcgr
                ).unsqueeze(0),

            "class_id":
                torch.tensor(
                    self.labels[index],
                    dtype=torch.long
                )
        }


train_dataset = SingleFCGRDataset(

    train_metadata,

    fcgr_memmap,

    id_to_fcgr_row
)


val_dataset = SingleFCGRDataset(

    val_metadata,

    fcgr_memmap,

    id_to_fcgr_row
)


train_generator = torch.Generator()

train_generator.manual_seed(
    RANDOM_STATE
)


train_loader = DataLoader(

    train_dataset,

    batch_size=
        TRAIN_BATCH_SIZE,

    shuffle=True,

    generator=
        train_generator,

    num_workers=0,

    pin_memory=
        torch.cuda.is_available(),

    drop_last=False
)


val_loader = DataLoader(

    val_dataset,

    batch_size=
        EVAL_BATCH_SIZE,

    shuffle=False,

    num_workers=0,

    pin_memory=
        torch.cuda.is_available()
)


print(
    "Train samples/epoch:",
    len(train_dataset)
)

print(
    "Train batches:",
    len(train_loader)
)

print(
    "Validation:",
    len(val_dataset)
)


batch_check = next(
    iter(train_loader)
)


unique_classes, batch_counts = torch.unique(

    batch_check["class_id"],

    return_counts=True
)


print()
print(
    "Primo batch size:",
    len(batch_check["class_id"])
)

print(
    "Classi presenti:",
    unique_classes.tolist()
)

print(
    "Conteggi:",
    batch_counts.tolist()
)

Train samples/epoch: 96167
Train batches: 752
Validation: 9753

Primo batch size: 128
Classi presenti: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11]
Conteggi: [9, 15, 13, 12, 12, 11, 17, 20, 7, 7, 4, 1]


In [4]:
# ============================================================
# CELL 5 — COSINE PROXY HEAD
# ============================================================

class CosineProxyHead(nn.Module):

    def __init__(
        self,
        embedding_dim,
        n_classes,
        scale=16.0
    ):

        super().__init__()


        self.weight = nn.Parameter(

            torch.empty(
                n_classes,
                embedding_dim
            )
        )


        nn.init.xavier_uniform_(
            self.weight
        )


        self.scale = float(
            scale
        )


    def forward(
        self,
        embeddings
    ):

        # embedding già normalizzato
        proxies = F.normalize(

            self.weight,

            p=2,

            dim=1,

            eps=1e-8
        )


        cosine_similarity = (

            embeddings

            @

            proxies.T
        )


        logits = (
            self.scale
            *
            cosine_similarity
        )


        return (
            logits,
            cosine_similarity
        )


    def normalized_proxies(
        self
    ):

        return F.normalize(

            self.weight,

            p=2,

            dim=1,

            eps=1e-8
        )

In [5]:
# ============================================================
# CELL 6 — WEIGHTED SUPCON PROXY V3
# ============================================================

class WeightedSupConProxyV3(nn.Module):

    def __init__(
        self,
        embedding_dim=128,
        projection_dim=128,
        n_classes=12,
        proxy_scale=16.0
    ):

        super().__init__()


        # ====================================================
        # CNN V3
        # ====================================================

        self.features = nn.Sequential(

            nn.Conv2d(
                1, 32, 3,
                padding=1,
                bias=False
            ),

            nn.GroupNorm(8, 32),
            nn.ReLU(inplace=True),

            nn.Conv2d(
                32, 32, 3,
                padding=1,
                bias=False
            ),

            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            nn.Conv2d(
                32, 64, 3,
                padding=1,
                bias=False
            ),

            nn.GroupNorm(8, 64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            nn.Conv2d(
                64, 128, 3,
                padding=1,
                bias=False
            ),

            nn.GroupNorm(8, 128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            nn.Conv2d(
                128, 128, 3,
                padding=1,
                bias=False
            ),

            nn.GroupNorm(8, 128),
            nn.ReLU(inplace=True),

            nn.AdaptiveAvgPool2d(
                (4, 4)
            )
        )


        # ====================================================
        # EMBEDDING h
        # ====================================================

        self.embedding_head = nn.Sequential(

            nn.Flatten(),

            nn.Linear(
                128 * 4 * 4,
                256
            ),

            nn.ReLU(inplace=True),

            nn.Linear(
                256,
                embedding_dim
            )
        )


        # ====================================================
        # PROJECTION z
        # ====================================================

        self.projection_head = nn.Sequential(

            nn.Linear(
                embedding_dim,
                embedding_dim
            ),

            nn.ReLU(inplace=True),

            nn.Linear(
                embedding_dim,
                projection_dim
            )
        )


        # ====================================================
        # LEARNED CLASS PROXIES
        # ====================================================

        self.proxy_head = CosineProxyHead(

            embedding_dim=
                embedding_dim,

            n_classes=
                n_classes,

            scale=
                proxy_scale
        )


    def forward(
        self,
        x
    ):

        features = self.features(
            x
        )


        h_raw = self.embedding_head(
            features
        )


        h = F.normalize(
            h_raw,
            p=2,
            dim=1,
            eps=1e-8
        )


        z_raw = self.projection_head(
            h_raw
        )


        z = F.normalize(
            z_raw,
            p=2,
            dim=1,
            eps=1e-8
        )


        (
            logits,
            proxy_similarities
        ) = self.proxy_head(
            h
        )


        return (
            h,
            z,
            logits,
            proxy_similarities
        )


model = WeightedSupConProxyV3(

    embedding_dim=
        EMBEDDING_DIM,

    projection_dim=
        PROJECTION_DIM,

    n_classes=
        N_CLASSES,

    proxy_scale=
        PROXY_SCALE

).to(
    DEVICE
)

In [6]:
# ============================================================
# CELL 7 — LOAD PRETRAINED SUPCON
# ============================================================

supcon_checkpoint = torch.load(

    SUPCON_CHECKPOINT_PATH,

    map_location=DEVICE
)


load_result = model.load_state_dict(

    supcon_checkpoint[
        "model_state_dict"
    ],

    strict=False
)


print("=" * 72)
print("SUPCON PRETRAINED LOAD")
print("=" * 72)

print(
    "Source epoch:",
    supcon_checkpoint[
        "best_epoch"
    ]
)

print(
    "Source Macro-F1:",
    supcon_checkpoint[
        "best_val_macro_f1"
    ]
)

print()

print(
    "Missing keys:",
    load_result.missing_keys
)

print(
    "Unexpected keys:",
    load_result.unexpected_keys
)


assert (
    load_result.unexpected_keys
    ==
    []
)


assert (
    load_result.missing_keys
    ==
    ["proxy_head.weight"]
)


print()
print(
    "SupCon encoder + projection caricati: OK"
)

print(
    "Proxy head inizializzato da zero: OK"
)

SUPCON PRETRAINED LOAD
Source epoch: 26
Source Macro-F1: 0.27066922265712795

Missing keys: ['proxy_head.weight']
Unexpected keys: []

SupCon encoder + projection caricati: OK
Proxy head inizializzato da zero: OK


In [7]:
# ============================================================
# CELL 8 — FLEXIBLE SUPERVISED CONTRASTIVE LOSS
# ============================================================

class FlexibleSupervisedContrastiveLoss(nn.Module):

    def __init__(
        self,
        temperature=0.07
    ):

        super().__init__()

        self.temperature = float(
            temperature
        )


    def forward(
        self,
        projections,
        labels
    ):

        projections = (
            projections.float()
        )

        labels = (
            labels.long()
        )


        batch_size = (
            projections.shape[0]
        )


        # ====================================================
        # COSINE LOGITS
        # ====================================================

        cosine_matrix = (

            projections

            @

            projections.T
        )


        logits = (
            cosine_matrix
            /
            self.temperature
        )


        logits = (

            logits

            -

            logits.max(
                dim=1,
                keepdim=True
            ).values.detach()
        )


        identity = torch.eye(

            batch_size,

            dtype=torch.bool,

            device=labels.device
        )


        same_class = (

            labels.unsqueeze(0)

            ==

            labels.unsqueeze(1)
        )


        positive_mask = (

            same_class

            &

            ~identity
        )


        denominator_mask = (
            ~identity
        )


        exp_logits = (

            torch.exp(
                logits
            )

            *
            denominator_mask.float()
        )


        log_prob = (

            logits

            -

            torch.log(

                exp_logits
                .sum(
                    dim=1,
                    keepdim=True
                )

                +
                1e-12
            )
        )


        positives_per_anchor = (
            positive_mask
            .sum(dim=1)
        )


        valid_anchor = (
            positives_per_anchor
            >
            0
        )


        mean_log_prob_positive = (

            (
                positive_mask.float()

                *

                log_prob
            )
            .sum(dim=1)

            /

            positives_per_anchor
            .clamp_min(1)
        )


        if valid_anchor.any():

            loss = (

                -mean_log_prob_positive[
                    valid_anchor
                ]
                .mean()
            )

        else:

            loss = (
                projections.sum()
                *
                0.0
            )


        # ====================================================
        # DIAGNOSTICS
        # ====================================================

        negative_mask = (
            ~same_class
        )


        if positive_mask.any():

            positive_cosine = (

                cosine_matrix[
                    positive_mask
                ]
                .mean()
            )

        else:

            positive_cosine = torch.tensor(

                float("nan"),

                device=
                    projections.device
            )


        negative_cosine = (

            cosine_matrix[
                negative_mask
            ]
            .mean()
        )


        valid_anchor_fraction = (

            valid_anchor
            .float()
            .mean()
        )


        return (

            loss,

            {
                "positive_cosine":
                    positive_cosine.detach(),

                "negative_cosine":
                    negative_cosine.detach(),

                "cosine_gap":
                    (
                        positive_cosine
                        -
                        negative_cosine
                    ).detach(),

                "valid_anchor_fraction":
                    valid_anchor_fraction.detach()
            }
        )


supcon_criterion = (
    FlexibleSupervisedContrastiveLoss(
        temperature=
            TEMPERATURE
    )
)


weighted_ce_criterion = nn.CrossEntropyLoss(

    weight=
        class_weights_tensor
)

In [10]:
# ============================================================
# CELL 9 — WEIGHTED CE + SUPCON
# ============================================================

class CombinedMetricLoss(nn.Module):

    def __init__(
        self,
        ce_criterion,
        supcon_criterion,
        supcon_weight=0.25
    ):

        super().__init__()

        self.ce_criterion = (
            ce_criterion
        )

        self.supcon_criterion = (
            supcon_criterion
        )

        self.supcon_weight = float(
            supcon_weight
        )


    def forward(
        self,
        logits,
        projections,
        labels
    ):

        # ====================================================
        # WEIGHTED CE IN FP32
        #
        # Con AMP i logits possono essere float16, mentre
        # i class weights sono float32.
        # Calcoliamo quindi esplicitamente la CE in FP32.
        # ====================================================

        ce_loss = self.ce_criterion(

            logits.float(),

            labels
        )


        # ====================================================
        # SUPCON
        #
        # Anche FlexibleSupervisedContrastiveLoss converte
        # internamente projections in FP32.
        # ====================================================

        (
            supcon_loss,
            supcon_stats
        ) = self.supcon_criterion(

            projections,

            labels
        )


        # ====================================================
        # COMBINED LOSS
        # ====================================================

        total_loss = (

            ce_loss

            +

            self.supcon_weight
            *
            supcon_loss
        )


        stats = {

            "ce_loss":
                ce_loss.detach(),

            "supcon_loss":
                supcon_loss.detach(),

            **supcon_stats
        }


        return (
            total_loss,
            stats
        )


combined_criterion = CombinedMetricLoss(

    ce_criterion=
        weighted_ce_criterion,

    supcon_criterion=
        supcon_criterion,

    supcon_weight=
        SUPCON_WEIGHT
)


print(
    "Combined loss configurata:"
)

print(
    f"  Total = WeightedCE + {SUPCON_WEIGHT} × SupCon"
)

print(
    "  Weighted CE dtype: FP32"
)

print(
    "  SupCon dtype: FP32"
)

Combined loss configurata:
  Total = WeightedCE + 0.25 × SupCon
  Weighted CE dtype: FP32
  SupCon dtype: FP32


In [11]:
# ============================================================
# CELL 10 — FULL SANITY CHECK
# ============================================================

model.eval()


batch_check = next(
    iter(train_loader)
)


x = (
    batch_check["x"]
    .to(
        DEVICE,
        non_blocking=True
    )
)


labels = (
    batch_check["class_id"]
    .to(
        DEVICE,
        non_blocking=True
    )
)


with torch.no_grad():

    with torch.autocast(

        device_type=
            DEVICE.type,

        dtype=(
            torch.float16
            if DEVICE.type == "cuda"
            else torch.bfloat16
        ),

        enabled=
            AMP_ENABLED

    ):

        (
            h,
            z,
            logits,
            proxy_similarities
        ) = model(
            x
        )


    (
        sanity_loss,
        sanity_stats
    ) = combined_criterion(

        logits,
        z,
        labels
    )


predictions = (
    logits.argmax(
        dim=1
    )
)


batch_accuracy = (

    predictions
    .eq(labels)
    .float()
    .mean()
    .item()
)


unique_classes, counts = torch.unique(

    labels,

    return_counts=True
)


print("=" * 78)
print("WEIGHTED SUPCON + PROXY — INITIAL SANITY CHECK")
print("=" * 78)

print(
    "Batch size:",
    len(labels)
)

print(
    "Classi presenti:",
    len(unique_classes)
)

print(
    "Conteggi:",
    counts.tolist()
)

print()

print(
    "Total loss:",
    f"{sanity_loss.item():.4f}"
)

print(
    "Weighted CE:",
    f"{sanity_stats['ce_loss'].item():.4f}"
)

print(
    "SupCon:",
    f"{sanity_stats['supcon_loss'].item():.4f}"
)

print()

print(
    "Valid SupCon anchors:",
    f"{sanity_stats['valid_anchor_fraction'].item() * 100:.2f}%"
)

print(
    "Positive cosine:",
    f"{sanity_stats['positive_cosine'].item():.4f}"
)

print(
    "Negative cosine:",
    f"{sanity_stats['negative_cosine'].item():.4f}"
)

print(
    "Cosine gap:",
    f"{sanity_stats['cosine_gap'].item():.4f}"
)

print()

print(
    "Initial proxy batch accuracy:",
    f"{batch_accuracy:.4f}"
)

print(
    "h norm:",
    f"{h.norm(dim=1).mean().item():.4f}"
)

print(
    "z norm:",
    f"{z.norm(dim=1).mean().item():.4f}"
)


assert torch.isfinite(
    sanity_loss
)


assert (
    sanity_stats[
        "valid_anchor_fraction"
    ].item()
    >
    0.5
)


print()
print(
    "Weighted SupCon + Proxy sanity check: OK"
)

WEIGHTED SUPCON + PROXY — INITIAL SANITY CHECK
Batch size: 128
Classi presenti: 12
Conteggi: [10, 8, 16, 17, 14, 11, 23, 17, 3, 5, 3, 1]

Total loss: 4.6145
Weighted CE: 3.4354
SupCon: 4.7164

Valid SupCon anchors: 99.22%
Positive cosine: 0.9348
Negative cosine: 0.9037
Cosine gap: 0.0311

Initial proxy batch accuracy: 0.1172
h norm: 1.0000
z norm: 1.0000

Weighted SupCon + Proxy sanity check: OK


In [12]:
# ============================================================
# CELL 11 — PROXY-BASED MULTICLASS EVALUATION
# ============================================================

def evaluate_proxy_classifier(
    model,
    loader
):

    model.eval()

    all_y_true = []
    all_y_pred = []

    all_max_similarity = []


    with torch.no_grad():

        for batch in loader:

            x = (
                batch["x"]
                .to(
                    DEVICE,
                    non_blocking=True
                )
            )

            labels = (
                batch["class_id"]
                .to(
                    DEVICE,
                    non_blocking=True
                )
            )


            with torch.autocast(

                device_type=DEVICE.type,

                dtype=(
                    torch.float16
                    if DEVICE.type == "cuda"
                    else torch.bfloat16
                ),

                enabled=AMP_ENABLED
            ):

                (
                    h,
                    z,
                    logits,
                    proxy_similarities
                ) = model(x)


            predictions = (
                proxy_similarities
                .argmax(dim=1)
            )


            max_similarity = (
                proxy_similarities
                .max(dim=1)
                .values
            )


            all_y_true.append(
                labels.cpu().numpy()
            )

            all_y_pred.append(
                predictions.cpu().numpy()
            )

            all_max_similarity.append(
                max_similarity
                .float()
                .cpu()
                .numpy()
            )


    y_true = np.concatenate(
        all_y_true
    )

    y_pred = np.concatenate(
        all_y_pred
    )

    max_similarity = np.concatenate(
        all_max_similarity
    )


    metrics = {

        "accuracy":
            float(
                accuracy_score(
                    y_true,
                    y_pred
                )
            ),

        "macro_f1":
            float(
                f1_score(
                    y_true,
                    y_pred,
                    average="macro",
                    zero_division=0
                )
            ),

        "balanced_accuracy":
            float(
                balanced_accuracy_score(
                    y_true,
                    y_pred
                )
            ),

        "mean_max_similarity":
            float(
                max_similarity.mean()
            )
    }


    return (
        metrics,
        y_true,
        y_pred
    )

In [13]:
# ============================================================
# CELL 12 — RANDOM PROXY BASELINE
# ============================================================

initial_04j_state = copy.deepcopy(
    model.state_dict()
)


(
    initial_proxy_metrics,
    _,
    _
) = evaluate_proxy_classifier(

    model,
    val_loader
)


print("=" * 72)
print("04j — BEFORE PROXY TRAINING")
print("=" * 72)

print(
    "Accuracy:",
    f"{initial_proxy_metrics['accuracy']:.4f}"
)

print(
    "Macro-F1:",
    f"{initial_proxy_metrics['macro_f1']:.4f}"
)

print(
    "Balanced Accuracy:",
    f"{initial_proxy_metrics['balanced_accuracy']:.4f}"
)

print(
    "Mean max similarity:",
    f"{initial_proxy_metrics['mean_max_similarity']:.4f}"
)

04j — BEFORE PROXY TRAINING
Accuracy: 0.0949
Macro-F1: 0.0397
Balanced Accuracy: 0.0772
Mean max similarity: 0.1691


In [14]:
# ============================================================
# CELL 13 — PROXY-ONLY WARM-UP
# ============================================================

# ------------------------------------------------------------
# Freeze everything
# ------------------------------------------------------------

for parameter in model.parameters():

    parameter.requires_grad = False


# ------------------------------------------------------------
# Unfreeze proxy only
# ------------------------------------------------------------

for parameter in model.proxy_head.parameters():

    parameter.requires_grad = True


proxy_warmup_optimizer = torch.optim.AdamW(

    model.proxy_head.parameters(),

    lr=PROXY_LR,

    weight_decay=WEIGHT_DECAY
)


warmup_scaler = torch.amp.GradScaler(

    "cuda",

    enabled=AMP_ENABLED
)


model.train()


total_ce = 0.0
total_correct = 0
total_samples = 0


start_time = time.perf_counter()


for batch in train_loader:

    x = (
        batch["x"]
        .to(
            DEVICE,
            non_blocking=True
        )
    )


    labels = (
        batch["class_id"]
        .to(
            DEVICE,
            non_blocking=True
        )
    )


    proxy_warmup_optimizer.zero_grad(
        set_to_none=True
    )


    with torch.autocast(

        device_type=DEVICE.type,

        dtype=(
            torch.float16
            if DEVICE.type == "cuda"
            else torch.bfloat16
        ),

        enabled=AMP_ENABLED
    ):

        (
            h,
            z,
            logits,
            _
        ) = model(x)


    # Weighted CE in FP32
    ce_loss = weighted_ce_criterion(

        logits.float(),

        labels
    )


    if AMP_ENABLED:

        warmup_scaler.scale(
            ce_loss
        ).backward()

        warmup_scaler.step(
            proxy_warmup_optimizer
        )

        warmup_scaler.update()

    else:

        ce_loss.backward()

        proxy_warmup_optimizer.step()


    predictions = (
        logits
        .detach()
        .argmax(dim=1)
    )


    batch_size = labels.shape[0]


    total_ce += (
        ce_loss.item()
        *
        batch_size
    )


    total_correct += (
        predictions
        .eq(labels)
        .sum()
        .item()
    )


    total_samples += (
        batch_size
    )


elapsed = (
    time.perf_counter()
    -
    start_time
)


warmup_train_ce = (
    total_ce
    /
    total_samples
)


warmup_train_accuracy = (
    total_correct
    /
    total_samples
)


# ============================================================
# VALIDATION
# ============================================================

(
    warmup_val_metrics,
    _,
    _
) = evaluate_proxy_classifier(

    model,
    val_loader
)


print("=" * 80)
print("PROXY-ONLY WARM-UP — 1 EPOCH")
print("=" * 80)

print(
    "Train Weighted CE:",
    f"{warmup_train_ce:.4f}"
)

print(
    "Train Accuracy:",
    f"{warmup_train_accuracy:.4f}"
)

print()

print(
    "Val Accuracy:",
    f"{warmup_val_metrics['accuracy']:.4f}"
)

print(
    "Val Macro-F1:",
    f"{warmup_val_metrics['macro_f1']:.4f}"
)

print(
    "Val Balanced Accuracy:",
    f"{warmup_val_metrics['balanced_accuracy']:.4f}"
)

print()

print(
    "Seconds:",
    f"{elapsed:.1f}"
)

PROXY-ONLY WARM-UP — 1 EPOCH
Train Weighted CE: 1.8478
Train Accuracy: 0.3208

Val Accuracy: 0.3170
Val Macro-F1: 0.2788
Val Balanced Accuracy: 0.3041

Seconds: 10.6


In [15]:
# ============================================================
# CELL 14 — SAVE POST-WARMUP STATE + UNFREEZE
# ============================================================

post_warmup_state = copy.deepcopy(
    model.state_dict()
)


# Tutto nuovamente trainable
for parameter in model.parameters():

    parameter.requires_grad = True


backbone_parameters = []

proxy_parameters = []


for name, parameter in model.named_parameters():

    if name.startswith(
        "proxy_head."
    ):

        proxy_parameters.append(
            parameter
        )

    else:

        backbone_parameters.append(
            parameter
        )


print(
    "Backbone parameters:",
    sum(
        p.numel()
        for p in backbone_parameters
    )
)

print(
    "Proxy parameters:",
    sum(
        p.numel()
        for p in proxy_parameters
    )
)

Backbone parameters: 840288
Proxy parameters: 1536


In [16]:
# ============================================================
# CELL 15 — JOINT OPTIMIZER
# ============================================================

optimizer = torch.optim.AdamW(

    [
        {
            "params":
                backbone_parameters,

            "lr":
                BACKBONE_LR
        },

        {
            "params":
                proxy_parameters,

            "lr":
                PROXY_LR
        }
    ],

    weight_decay=
        WEIGHT_DECAY
)


scaler = torch.amp.GradScaler(

    "cuda",

    enabled=AMP_ENABLED
)


print(
    "Backbone LR:",
    BACKBONE_LR
)

print(
    "Proxy LR:",
    PROXY_LR
)

print(
    "Weight decay:",
    WEIGHT_DECAY
)

print(
    "SupCon weight:",
    SUPCON_WEIGHT
)

Backbone LR: 0.0001
Proxy LR: 0.0005
Weight decay: 0.0001
SupCon weight: 0.25


In [17]:
# ============================================================
# CELL 16 — TRAIN ONE WEIGHTED SUPCON + PROXY EPOCH
# ============================================================

def train_joint_epoch(
    model,
    loader,
    criterion,
    optimizer,
    scaler
):

    model.train()


    total_loss = 0.0

    total_ce = 0.0

    total_supcon = 0.0

    total_positive_cosine = 0.0

    total_negative_cosine = 0.0

    total_gap = 0.0

    total_valid_anchor = 0.0

    total_h_std = 0.0


    total_correct = 0

    total_samples = 0

    total_batches = 0


    start_time = time.perf_counter()


    for batch in loader:

        x = (
            batch["x"]
            .to(
                DEVICE,
                non_blocking=True
            )
        )


        labels = (
            batch["class_id"]
            .to(
                DEVICE,
                non_blocking=True
            )
        )


        optimizer.zero_grad(
            set_to_none=True
        )


        with torch.autocast(

            device_type=DEVICE.type,

            dtype=(
                torch.float16
                if DEVICE.type == "cuda"
                else torch.bfloat16
            ),

            enabled=AMP_ENABLED

        ):

            (
                h,
                z,
                logits,
                proxy_similarities
            ) = model(x)


        (
            loss,
            stats
        ) = criterion(

            logits,
            z,
            labels
        )


        if AMP_ENABLED:

            scaler.scale(
                loss
            ).backward()


            scaler.step(
                optimizer
            )


            scaler.update()

        else:

            loss.backward()

            optimizer.step()


        # ====================================================
        # METRICS
        # ====================================================

        batch_size = (
            labels.shape[0]
        )


        predictions = (
            proxy_similarities
            .detach()
            .argmax(dim=1)
        )


        total_correct += (

            predictions
            .eq(labels)
            .sum()
            .item()
        )


        total_samples += (
            batch_size
        )


        total_loss += (
            loss.item()
        )


        total_ce += (
            stats[
                "ce_loss"
            ].item()
        )


        total_supcon += (
            stats[
                "supcon_loss"
            ].item()
        )


        total_positive_cosine += (
            stats[
                "positive_cosine"
            ].item()
        )


        total_negative_cosine += (
            stats[
                "negative_cosine"
            ].item()
        )


        total_gap += (
            stats[
                "cosine_gap"
            ].item()
        )


        total_valid_anchor += (
            stats[
                "valid_anchor_fraction"
            ].item()
        )


        total_h_std += (

            h.float()
            .std(
                dim=0
            )
            .mean()
            .item()
        )


        total_batches += 1


    elapsed = (
        time.perf_counter()
        -
        start_time
    )


    return {

        "loss":
            total_loss
            /
            total_batches,

        "ce_loss":
            total_ce
            /
            total_batches,

        "supcon_loss":
            total_supcon
            /
            total_batches,

        "positive_cosine":
            total_positive_cosine
            /
            total_batches,

        "negative_cosine":
            total_negative_cosine
            /
            total_batches,

        "cosine_gap":
            total_gap
            /
            total_batches,

        "valid_anchor_fraction":
            total_valid_anchor
            /
            total_batches,

        "h_std":
            total_h_std
            /
            total_batches,

        "proxy_accuracy":
            total_correct
            /
            total_samples,

        "seconds":
            elapsed
    }

In [18]:
# ============================================================
# CELL 17 — 04j SMOKE TEST
# ============================================================

SMOKE_EPOCHS = 3


print("=" * 108)
print("WEIGHTED SUPCON + LEARNED PROXIES — SMOKE TEST")
print("=" * 108)


for epoch in range(
    1,
    SMOKE_EPOCHS + 1
):

    train_metrics = train_joint_epoch(

        model=
            model,

        loader=
            train_loader,

        criterion=
            combined_criterion,

        optimizer=
            optimizer,

        scaler=
            scaler
    )


    (
        val_metrics,
        _,
        _
    ) = evaluate_proxy_classifier(

        model,
        val_loader
    )


    print(

        f"Epoch {epoch:02d}/{SMOKE_EPOCHS}"

        f" | total "
        f"{train_metrics['loss']:.4f}"

        f" | CE "
        f"{train_metrics['ce_loss']:.4f}"

        f" | SupCon "
        f"{train_metrics['supcon_loss']:.4f}"

        f" | gap "
        f"{train_metrics['cosine_gap']:.4f}"

        f" | h_std "
        f"{train_metrics['h_std']:.5f}"

        f" | anchors "
        f"{train_metrics['valid_anchor_fraction'] * 100:.1f}%"

        f" | train proxy Acc "
        f"{train_metrics['proxy_accuracy']:.4f}"

        f" | val Acc "
        f"{val_metrics['accuracy']:.4f}"

        f" | val F1 "
        f"{val_metrics['macro_f1']:.4f}"

        f" | bal Acc "
        f"{val_metrics['balanced_accuracy']:.4f}"

        f" | "
        f"{train_metrics['seconds']:.1f}s"
    )

WEIGHTED SUPCON + LEARNED PROXIES — SMOKE TEST
Epoch 01/3 | total 2.8156 | CE 1.6801 | SupCon 4.5423 | gap 0.0514 | h_std 0.05399 | anchors 99.8% | train proxy Acc 0.3661 | val Acc 0.3106 | val F1 0.2864 | bal Acc 0.3055 | 57.4s
Epoch 02/3 | total 2.7692 | CE 1.6400 | SupCon 4.5165 | gap 0.0565 | h_std 0.05361 | anchors 99.8% | train proxy Acc 0.3774 | val Acc 0.3260 | val F1 0.2901 | bal Acc 0.3109 | 24.9s
Epoch 03/3 | total 2.7361 | CE 1.6119 | SupCon 4.4967 | gap 0.0609 | h_std 0.05390 | anchors 99.8% | train proxy Acc 0.3841 | val Acc 0.3199 | val F1 0.2908 | bal Acc 0.3109 | 24.6s
